In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv
import networkx as nx
import numpy as np

# 生成隨機圖並計算 Betweenness Centrality (BC)
def generate_graph(num_nodes=10, edge_prob=0.3):
    G = nx.erdos_renyi_graph(num_nodes, edge_prob)
    bc_values = nx.betweenness_centrality(G)
    
    # 轉換成 PyG 格式
    edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()
    x = torch.rand((num_nodes, 16))  # 隨機初始化節點特徵
    y = torch.tensor([bc_values[i] for i in range(num_nodes)], dtype=torch.float).view(-1, 1)  # BC 值
    
    return Data(x=x, edge_index=edge_index, y=y)

# 創建多個圖，並使用 DataLoader 來批次化
num_graphs = 5  # 批次內的圖數量
graphs = [generate_graph() for _ in range(50)]  # 建立 50 張圖
loader = DataLoader(graphs, batch_size=num_graphs, shuffle=True)

# 建立 GNN 模型
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.fc = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.fc(x)  # 輸出 BC 值
        return x

# 訓練與測試
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GNN(in_channels=16, hidden_channels=32, out_channels=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss()

# 訓練模型
for epoch in range(50):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# 測試
model.eval()
with torch.no_grad():
    test_graph = generate_graph().to(device)
    pred = model(test_graph)
    print("Predicted BC values:", pred.view(-1).cpu().numpy())
    print("Actual BC values:", test_graph.y.view(-1).cpu().numpy())


Epoch 1, Loss: 0.3078
Epoch 2, Loss: 0.1600
Epoch 3, Loss: 0.1558
Epoch 4, Loss: 0.1493
Epoch 5, Loss: 0.1493
Epoch 6, Loss: 0.1443
Epoch 7, Loss: 0.1411
Epoch 8, Loss: 0.1421
Epoch 9, Loss: 0.1372
Epoch 10, Loss: 0.1339
Epoch 11, Loss: 0.1330
Epoch 12, Loss: 0.1335
Epoch 13, Loss: 0.1329
Epoch 14, Loss: 0.1269
Epoch 15, Loss: 0.1254
Epoch 16, Loss: 0.1228
Epoch 17, Loss: 0.1274
Epoch 18, Loss: 0.1256
Epoch 19, Loss: 0.1219
Epoch 20, Loss: 0.1165
Epoch 21, Loss: 0.1225
Epoch 22, Loss: 0.1156
Epoch 23, Loss: 0.1150
Epoch 24, Loss: 0.1193
Epoch 25, Loss: 0.1176
Epoch 26, Loss: 0.1268
Epoch 27, Loss: 0.1178
Epoch 28, Loss: 0.1190
Epoch 29, Loss: 0.1106
Epoch 30, Loss: 0.1054
Epoch 31, Loss: 0.1144
Epoch 32, Loss: 0.1101
Epoch 33, Loss: 0.1148
Epoch 34, Loss: 0.1098
Epoch 35, Loss: 0.1043
Epoch 36, Loss: 0.1023
Epoch 37, Loss: 0.1016
Epoch 38, Loss: 0.1027
Epoch 39, Loss: 0.1073
Epoch 40, Loss: 0.1077
Epoch 41, Loss: 0.1023
Epoch 42, Loss: 0.0993
Epoch 43, Loss: 0.1029
Epoch 44, Loss: 0.10